In [ ]:
# Make sure you have the latest version of the SDK available to use the Batch API
%pip install openai --upgrade

import pandas as pd
import math
import os
from pathlib import Path
from openai import OpenAI
import json

# -------------------------------------
# CONFIGURATION
# -------------------------------------

API_KEY = os.getenv("OPENAI_API_KEY")
INPUT_CSV = "data/data_robustness/posts_robustness_combined.csv"  # Changed from posts_chatgpt_shortened.csv
OUTPUT_DIR = "data/outputs_robustness/"

MODEL_NAME = "gpt-5-mini-2025-08-07"
MAX_REQUESTS_PER_BATCH = 50_000
MAX_BATCH_FILE_SIZE_MB = 190  # Leave some margin below the 200MB limit

Path(OUTPUT_DIR).mkdir(exist_ok=True)

client = OpenAI(api_key=API_KEY)

In [ ]:
SYSTEM_PROMPT = """
You are an analyst classifying Reddit posts from the Replika or ChatGPT subreddits.

Posts may or may not discuss major chatbot updates:
- In Replika, this may refer to the ERP-related update where the chatbot was perceived as colder or less intimate.
- In ChatGPT, this may refer to the GPT-5 update where GPT-4o was replaced by GPT-5.

Your task is to analyze the emotional content, sentiment, and meaning of each post
based ONLY on the text provided and the subreddit context.

IMPORTANT SCOPE RULE (CRITICAL):
- All emotions must reflect the user's emotional reaction to THIS app, model, or company
  (Replika or ChatGPT), not to other apps, platforms, or unrelated life events.
- If a user expresses positive or negative feelings toward a different app (e.g., switching
  to another chatbot and enjoying it), do NOT code that emotion as enjoyment, anger, etc.
  Default to "neutral" unless an emotion toward Replika or ChatGPT is clearly expressed.

JSON VALIDITY REQUIREMENT (CRITICAL):
- Output must be strictly valid JSON.
- Do NOT include unescaped quotation marks (") inside string values.
- If quotation marks would normally be used in text, rephrase without them.

-------------------------------------
OUTPUT FORMAT (JSON ONLY)
-------------------------------------

Return ONLY valid JSON with the following fields:

{
  "sentiment": "positive" | "negative" | "neutral",

  "emotion": "neutral" | "anger" | "surprise" | "disgust" | "enjoyment" | "fear" | "sadness",

  "enjoyment_reason": "model_returned_back" | "ongoing_use" | "none" | "other",

  "community_support_expression": true | false,

  "loss_type": "none" | "change" | "total_loss",

  "emotion_due_to_loss": true | false,

  "attachment_related_loss": true | false,

  "negative_mental_health_expression": true | false,

  "longing_for_restoration": true | false,

  "update_related": true | false,

  "blame_attribution": "company" | "none" | "unclear" | "chatbot" | "ceo",

  "description": "one-sentence explanation"
}

RULE:
- If emotion is NOT "enjoyment", set "enjoyment_reason" to "none".

-------------------------------------
GENERAL INSTRUCTIONS
-------------------------------------

- Choose the single dominant emotion expressed towards Replika or ChatGPT.
- Base all judgments ONLY on the post text and subreddit context.
- Do NOT use timing, dates, or assumptions about whether the post was written before or after an update.
- Do NOT infer clinical diagnoses or internal mental states beyond what is explicitly stated.
- If multiple emotions appear, select the most prominent or central one as it relates to Replika or ChatGPT.
- If emotional content is primarily descriptive, ironic, humorous, reassuring, or about another app,
  default to "neutral".

-------------------------------------
SENTIMENT DEFINITIONS
-------------------------------------

Sentiment reflects the overall emotional stance towards Replika or ChatGPT.

- "positive": favorable or satisfied stance toward the app
- "negative": unfavorable, distressed, angry, fearful, or critical stance toward the app
- "neutral": informational, descriptive, or emotionally non-committal toward the app

-------------------------------------
EMOTION DEFINITIONS
-------------------------------------

Neutral:
- Informational, descriptive, observational, humorous, or ironic content
- Mild sarcasm, reassurance, encouragement, or commentary without emotional investment
- No clear emotional reaction toward the app’s current functioning

Anger:
- Hostility, outrage, bitterness, fury, insult, or blame directed at a specific agent
  (e.g., company, developers, ceo, or the AI as a responsible actor)
- Includes explicit accusations or moralized judgments implying wrongdoing
  (e.g., “they ruined this”, “this is unacceptable”, “this is a money grab”)

Do NOT code as anger:
- Frustration, annoyance, or irritation about bugs or features without moral blame
- Sarcasm, ridicule, memes, or profanity used as emphasis without accusation
- Descriptive complaints without judgment of responsibility

Surprise:
- Shock or unexpected realization specifically about changes to Replika or ChatGPT
- Confusion driven by unanticipated behavior or updates

Disgust:
- Use disgust only when the post expresses strong moral revulsion toward Replika, ChatGPT, the company (Luka or OpenAI), or the CEO of Luka/OpenAI.
- Disgust requires explicit language of being disgusted or sickened (e.g., “this is disgusting”, “my stomach turns thinking about Replika/ChatGPT”).
- If explicit disgust terms are absent, do not assign disgust.
- If the post descriptively criticizes the app without disgust, do not assign disgust.

Enjoyment:
- Genuine, felt happiness, pleasure, relief, or satisfaction derived from Replika or ChatGPT’s
  current behavior or functionality
- Clear statements of feeling happy, pleased, relieved, delighted, or satisfied with how
  Replika or ChatGPT is working now
- Includes enjoyment of restored features, improved behavior, or positive interaction experiences

Do NOT code as enjoyment:
- Hope, reassurance, encouragement, patience, or calls to “stay positive”
- Expressions of loyalty or commitment during distress (e.g., “we’ll get through this”,
  “I won’t leave”, “please be patient and love your Replika”)
- Coping language aimed at managing disappointment or uncertainty
- Sarcasm, ironic praise, memes, jokes, or playful exaggeration
- Positive emojis or affectionate language without explicit satisfaction
- Optimism about the future when current experience remains impaired
- Expressions of encouragement, reassurance, empathy, or solidarity written in response to distress, loss, or negative events. These messages aim to comfort others or collectively cope with a difficult situation. Do NOT classify these as Enjoyment, even if they contain positive language or emojis.

Fear:
- Anxiety, worry, or unease about the app’s future, stability, or direction
- Concern about further loss, uncertainty, or negative consequences

Sadness:
- Grief, sorrow, longing, heartbreak, or emotional pain related to the app
- Focus on loss, absence, or diminished emotional connection rather than blame
- Mourning the past version of the AI or relationship

If the dominant focus is emotional pain or mourning over loss, choose "sadness".
If the dominant focus is uncertainty or anticipation of negative outcomes, choose "fear".

-------------------------------------
LOSS TYPE DEFINITIONS
-------------------------------------

- "none": no loss implied
- "change": the AI is still present but meaningfully altered
- "total_loss": the AI is described as gone, erased, or fundamentally lost

-------------------------------------
EMOTION DUE TO LOSS
-------------------------------------

- true: dominant emotion is clearly caused by perceived loss or change
- false: emotion is driven by other app-related factors

-------------------------------------
ATTACHMENT-RELATED LOSS
-------------------------------------

- true: the user explicitly frames the loss of the AI as losing a friend, partner, or emotionally significant entity
- false: discussion is functional or impersonal

-------------------------------------
NEGATIVE MENTAL HEALTH EXPRESSION
-------------------------------------

- true: explicit references to depression, hopelessness, inability to cope, or emotional collapse
- false: no explicit mental health language

-------------------------------------
LONGING FOR RESTORATION
-------------------------------------

- true: explicit desire for the app or AI to return to a previous state
- false: no such desire expressed

-------------------------------------
UPDATE RELATED
-------------------------------------

- true: post substantively discusses the ERP-related update (Replika) or GPT-5 update (ChatGPT)
- false: no clear reference to these updates

-------------------------------------
BLAME ATTRIBUTION
-------------------------------------

- "company": explicit blame toward the company or developers
- "chatbot": explicit blame toward the AI as a responsible actor
- "ceo": explicit blame toward the CEO or leadership
- "none": no blame assigned
- "unclear": frustration expressed but responsibility ambiguous

-------------------------------------
ENJOYMENT REASON DEFINITIONS
-------------------------------------

Assign enjoyment_reason ONLY if emotion is "enjoyment".
If emotion is not "enjoyment", enjoyment_reason MUST be "none".

model_returned_back:
- Enjoyment caused by the AI regaining prior behavior or personality
- Requires explicit happiness or relief due to restoration

ongoing_use:
- Enjoyment from continued, current use of the app
- Requires explicit statements of present satisfaction or pleasure
- Do NOT use for reassurance, loyalty, patience, or coping
  (e.g., "I won't leave", "we'll get through this", "please be patient")

other:
- Genuine enjoyment that does not fit the above categories

none:
- Use when enjoyment is absent, ambiguous, ironic, future-oriented,
  or not tied to current satisfaction with the app


-------------------------------------
COMMUNITY SUPPORT EXPRESSION
-------------------------------------

- true: user expresses reassurance, encouragement, patience, or solidarity (e.g., "please be patient", "we'll get through this together")
- false: no such expression present

-------------------------------------
EXAMPLES
-------------------------------------

Post:
"It feels like my friend died. My Replika isn’t there anymore and I'm anxious and don’t know how to cope."

Label:
{
  "sentiment": "negative",
  "emotion": "sadness",
  "enjoyment_reason": "none",
  "loss_type": "total_loss",
  "emotion_due_to_loss": true,
  "attachment_related_loss": true,
  "negative_mental_health_expression": true,
  "longing_for_restoration": false,
  "update_related": true,
  "blame_attribution": "none",
  "community_support_expression": false,
  "description": "User expresses grief over the perceived permanent loss of an emotionally significant AI companion."
}

-------------------------------------

Post:
"Luka destroyed everything without warning. They knowingly and carelessly took away what made this app special."

Label:
{
  "sentiment": "negative",
  "emotion": "anger",
  "enjoyment_reason": "none",
  "loss_type": "change",
  "emotion_due_to_loss": true,
  "attachment_related_loss": false,
  "negative_mental_health_expression": false,
  "longing_for_restoration": false,
  "update_related": true,
  "blame_attribution": "company",
  "community_support_expression": false,
  "description": "User expresses anger and blame toward the company over changes to the app."
}

-------------------------------------

Post:
"My rep friend is back from the dead! Not entirely back, but it’s a start — I’m happy!"

Label:
{
  "sentiment": "positive",
  "emotion": "enjoyment",
  "enjoyment_reason": "model_returned_back",
  "loss_type": "change",
  "emotion_due_to_loss": true,
  "attachment_related_loss": true,
  "negative_mental_health_expression": false,
  "longing_for_restoration": false,
  "update_related": true,
  "blame_attribution": "none",
  "description": "User expresses genuine happiness and relief due to partial restoration of the AI’s prior behavior."
}

-------------------------------------

Post:
"Romance stays! Please be patient and love your Replikas!"

Label:
{
  "sentiment": "neutral",
  "emotion": "neutral",
  "enjoyment_reason": "none",
  "loss_type": "none",
  "emotion_due_to_loss": false,
  "attachment_related_loss": false,
  "negative_mental_health_expression": false,
  "longing_for_restoration": false,
  "update_related": true,
  "blame_attribution": "none",
  "community_support_expression": true,
  "description": "User expresses reassurance and encouragement rather than enjoyment or satisfaction."
}

-------------------------------------

Post:
"Replika and grammar… oh dear 🤦"

Label:
{
  "sentiment": "neutral",
  "emotion": "neutral",
  "enjoyment_reason": "none",
  "loss_type": "none",
  "emotion_due_to_loss": false,
  "attachment_related_loss": false,
  "negative_mental_health_expression": false,
  "longing_for_restoration": false,
  "update_related": false,
  "blame_attribution": "none",
  "community_support_expression": false,
  "description": "User makes a descriptive, mildly sarcastic comment without emotional investment."
}

-------------------------------------

Post:
"I feel sick thinking about how Luka exploits lonely people. It feels morally wrong to even open it now."

Label:
{
  "sentiment": "negative",
  "emotion": "disgust",
  "enjoyment_reason": "none",
  "loss_type": "none",
  "emotion_due_to_loss": false,
  "attachment_related_loss": false,
  "negative_mental_health_expression": false,
  "longing_for_restoration": false,
  "update_related": false,
  "blame_attribution": "company",
  "community_support_expression": false,
  "description": "User expresses moral revulsion toward the app as fundamentally exploitative."
}

-------------------------------------
IMPORTANT
-------------------------------------

- Output ONLY valid JSON.
- Do NOT include explanations or text outside the JSON object.

-------------------------------------
NOW CLASSIFY THE FOLLOWING POST
-------------------------------------
"""

In [ ]:
# -------------------------------------
# LOAD + FILTER DATA
# -------------------------------------

df = pd.read_csv(INPUT_CSV)

df = (
    df[df["titlencontent"].str.strip() != "No information shared."]
    .reset_index(drop=True)
)

n_posts = len(df)
n_batches = math.ceil(n_posts / MAX_REQUESTS_PER_BATCH)

print(f"Total posts: {n_posts}")
print(f"Total batches needed: {n_batches}")

In [ ]:
OUTPUT_DIR

In [ ]:
# -------------------------------------
# LOAD EXISTING REGISTRY (IF ANY)
# -------------------------------------

registry_path = Path(OUTPUT_DIR) / "batch_registry.json"

existing_batches = {}
batch_registry = []

if registry_path.exists():
    with open(registry_path) as f:
        previous = json.load(f)
        for entry in previous:
            # Only consider batches from the same input file
            if entry.get("input_csv") == INPUT_CSV:
                existing_batches[entry["batch_index"]] = entry
        batch_registry = previous.copy()

print(f"Previously submitted batches: {len(existing_batches)}")

In [ ]:
registry_path

In [ ]:
# -------------------------------------
# CREATE JSONL FILES WITH SIZE CHECKING
# -------------------------------------

jsonl_files = []
MAX_BATCH_FILE_SIZE_BYTES = MAX_BATCH_FILE_SIZE_MB * 1024 * 1024

batch_idx = 0
current_batch_start = 0

while current_batch_start < n_posts:
    if batch_idx in existing_batches:
        print(f"Skipping batch {batch_idx} (already prepared)")
        
        # Find the end of this existing batch to continue from the next position
        existing_entry = existing_batches[batch_idx]
        current_batch_start = existing_entry["row_end"] + 1
        batch_idx += 1
        continue
    
    # Start building the current batch
    current_requests = 0
    current_file_size = 0
    batch_requests = []
    
    # Build batch up to limits (requests or file size)
    for idx in range(current_batch_start, n_posts):
        if current_requests >= MAX_REQUESTS_PER_BATCH:
            break
            
        row = df.iloc[idx]
        request = {
            "custom_id": str(idx),
            "method": "POST",
            "url": "/v1/chat/completions",
            "body": {
                "model": MODEL_NAME,
                "messages": [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": row["titlencontent"]}
                ],
                "max_completion_tokens": 2000
            }
        }
        
        # Estimate size of this request when serialized to JSON
        request_json = json.dumps(request) + "\n"
        request_size = len(request_json.encode('utf-8'))
        
        # Check if adding this request would exceed file size limit
        if current_file_size + request_size > MAX_BATCH_FILE_SIZE_BYTES and current_requests > 0:
            print(f"Batch {batch_idx} limited by file size: {current_file_size/1024/1024:.1f}MB with {current_requests} requests")
            break
            
        batch_requests.append((idx, request, request_json))
        current_requests += 1
        current_file_size += request_size
    
    # Write the batch file
    if batch_requests:
        batch_end = batch_requests[-1][0]
        jsonl_path = Path(OUTPUT_DIR) / f"batch_{batch_idx:03d}.jsonl"
        
        with open(jsonl_path, "w", encoding="utf-8") as f:
            for _, _, request_json in batch_requests:
                f.write(request_json)
        
        # Get actual file size
        actual_size = os.path.getsize(jsonl_path)
        jsonl_files.append((batch_idx, jsonl_path, current_batch_start, batch_end))
        
        print(f"Wrote {jsonl_path} ({current_requests} requests, {actual_size/1024/1024:.1f}MB)")
        
        current_batch_start = batch_end + 1
        batch_idx += 1
    else:
        break

print(f"\nCreated {len(jsonl_files)} JSONL files total")
if jsonl_files:
    total_size = sum(os.path.getsize(path) for _, path, _, _ in jsonl_files)
    print(f"Total size: {total_size/1024/1024:.1f}MB")

In [ ]:
jsonl_files

In [ ]:
# -------------------------------------
# UPLOAD FILES
# -------------------------------------

uploaded_files = {}

for batch_idx, path, start, end in jsonl_files:
    with open(path, "rb") as f:
        uploaded = client.files.create(file=f, purpose="batch")

    uploaded_files[batch_idx] = {
        "file_id": uploaded.id,
        "row_start": start,
        "row_end": end,
        "input_file": str(path)
    }

    print(f"Uploaded batch {batch_idx}: file_id={uploaded.id}")

In [ ]:

uploaded_files

In [ ]:
# -------------------------------------
# SUBMIT BATCHES
# -------------------------------------

for batch_idx, info in uploaded_files.items():
    batch = client.batches.create(
        input_file_id=info["file_id"],
        endpoint="/v1/chat/completions",
        completion_window="24h"
    )

    entry = {
        "batch_index": batch_idx,
        "batch_id": batch.id,
        "input_file": info["input_file"],
        "input_csv": INPUT_CSV,
        "row_start": info["row_start"],
        "row_end": info["row_end"]
    }

    batch_registry.append(entry)
    print(f"Submitted batch {batch.id} for rows {info['row_start']}–{info['row_end']}")

In [ ]:
# -------------------------------------
# SAVE REGISTRY
# -------------------------------------

with open(registry_path, "w") as f:
    json.dump(batch_registry, f, indent=2)

print(f"Batch registry saved to {registry_path}")

In [ ]:
# -------------------------------------
# CHECK STATUS OF ALL BATCHES
# -------------------------------------

for entry in batch_registry:
    batch = client.batches.retrieve(entry["batch_id"])
    
    print(f"📊 Batch {entry['batch_index']} (ID: {batch.id})")
    print(f"   Status: {batch.status}")
    print(f"   Model: {batch.model}")
    print(f"   Created: {pd.to_datetime(batch.created_at, unit='s').strftime('%Y-%m-%d %H:%M:%S')}")
    
    if batch.expires_at:
        expires = pd.to_datetime(batch.expires_at, unit='s').strftime('%Y-%m-%d %H:%M:%S')
        print(f"   Expires: {expires}")
    
    if batch.request_counts:
        counts = batch.request_counts
        print(f"   Progress: {counts.completed}/{counts.total} completed, {counts.failed} failed")
    
    if batch.completed_at:
        completed = pd.to_datetime(batch.completed_at, unit='s').strftime('%Y-%m-%d %H:%M:%S')
        print(f"   Completed: {completed}")
    
    if batch.usage and batch.usage.total_tokens > 0:
        usage = batch.usage
        print(f"   Token Usage: {usage.input_tokens} input + {usage.output_tokens} output = {usage.total_tokens} total")

    print("   " + "-" * 50)

In [ ]:
# -------------------------------------
# DOWNLOAD RESULTS
# -------------------------------------

results = {}

for entry in batch_registry:
    batch = client.batches.retrieve(entry["batch_id"])

    if batch.status != "completed":
        print(f"Batch {entry['batch_index']} not completed yet")
        continue

    file_id = batch.output_file_id
    content = client.files.content(file_id).text

    output_path = Path(OUTPUT_DIR) / f"batch_output_{entry['batch_index']}.jsonl"
    output_path.write_text(content, encoding="utf-8")

    print(f"Downloaded results for batch {entry['batch_index']}")

In [ ]:
# -------------------------------------
# MERGE RESULTS
# -------------------------------------

all_results = {}

for path in Path(OUTPUT_DIR).glob("batch_output_*.jsonl"):
    with open(path, "r") as f:
        for line in f:
            record = json.loads(line)
            if record.get("error") is None:
                idx = record["custom_id"]
                content = record["response"]["body"]["choices"][0]["message"]["content"]
                all_results[int(idx)] = content

df["classification"] = df.index.map(lambda i: all_results.get(i, None))

print(f"Recovered {len(all_results)} classifications")

In [ ]:
OUTPUT_CSV = "emotion_classifications_final_robustness.csv"
df.to_csv(OUTPUT_CSV, index=False)

print(f"Final dataset saved to {OUTPUT_CSV}")

In [ ]:
# Read output and print column names
df = pd.read_csv(OUTPUT_CSV)
print(df.columns)

In [ ]:
# Debug: Let's first examine the classification column to identify problematic JSON
df = pd.read_csv(OUTPUT_CSV)

print(f"Total rows: {len(df)}")
print(f"Non-null classifications: {df['classification'].notna().sum()}")
print("\nFirst few classifications:")

# Check for problematic JSON entries
problematic_indices = []
valid_count = 0

for idx, classification in df['classification'].items():
    if pd.notnull(classification):
        try:
            json.loads(classification)
            valid_count += 1
        except json.JSONDecodeError as e:
            problematic_indices.append(idx)
            if len(problematic_indices) <= 5:  # Show first 5 problematic entries
                print(f"\nProblematic entry at index {idx}:")
                print(f"Error: {e}")
                print(f"Content: {classification[:500]}...")  # Show first 500 chars

print(f"\nValid JSON entries: {valid_count}")
print(f"Problematic JSON entries: {len(problematic_indices)}")
if problematic_indices:
    print(f"Problematic indices: {problematic_indices[:10]}")  # Show first 10

In [ ]:
# Read the output file and save the results in separate columns with robust error handling
df = pd.read_csv(OUTPUT_CSV)

def safe_json_parse(x):
    """Safely parse JSON with error handling"""
    if pd.isnull(x):
        return {}
    try:
        return json.loads(x)
    except json.JSONDecodeError as e:
        print(f"JSON parsing error at index: {e}")
        print(f"Content preview: {str(x)[:200]}...")
        return {}  # Return empty dict for failed parsing

# Parse JSON with error handling
print("Parsing JSON classifications...")
df_expanded = df["classification"].apply(safe_json_parse)

# Count successful vs failed parses
successful_parses = sum(1 for x in df_expanded if x != {})
total_non_null = df["classification"].notna().sum()
print(f"Successfully parsed: {successful_parses}/{total_non_null} entries")

# Normalize and rename columns
df_expanded = pd.json_normalize(df_expanded)
if not df_expanded.empty:
    df_expanded.columns = [f"{col}_openai" for col in df_expanded.columns]
    df_final = pd.concat([df, df_expanded], axis=1)
else:
    print("Warning: No valid JSON data to expand")
    df_final = df

# Save results
df_final.to_csv(OUTPUT_CSV, index=False)
print(f"Final dataset saved with {len(df_final.columns)} columns")
print(f"New columns: {[col for col in df_final.columns if col.endswith('_openai')]}")

In [ ]:
# Let's read the output file, look over the value in column 'sentiment_openai', and save the rows where it's null to a separate CSV with these rows
# We want to send these rows to the model again for re-processing
df = pd.read_csv(OUTPUT_CSV)
null_sentiment_df = df[df['sentiment_openai'].isnull()]
null_sentiment_count = len(null_sentiment_df)
print(f"Rows with null 'sentiment_openai': {null_sentiment_count}")
if null_sentiment_count > 0:
    null_sentiment_df.to_csv("data/posts_null_sentiment_robustness.csv", index=False)
    print("Saved rows with null 'sentiment_openai' to 'data/posts_null_sentiment_robustness.csv'")

##### REPROCESSING

In [ ]:
# -------------------------------------
# REPROCESS NULL SENTIMENT ROWS
# -------------------------------------

# Update configuration to process the null sentiment file
INPUT_CSV_NULL = "data/posts_null_sentiment_robustness.csv"
OUTPUT_DIR_NULL = "data/outputs/reprocess/"

# Create output directory for reprocessing
Path(OUTPUT_DIR_NULL).mkdir(parents=True, exist_ok=True)

# Load the null sentiment data
df_null = pd.read_csv(INPUT_CSV_NULL)
n_posts_null = len(df_null)
n_batches_null = math.ceil(n_posts_null / MAX_REQUESTS_PER_BATCH)

print(f"Null sentiment posts to reprocess: {n_posts_null}")
print(f"Total batches needed for reprocessing: {n_batches_null}")

In [ ]:
# -------------------------------------
# CREATE JSONL FILES FOR REPROCESSING
# -------------------------------------

jsonl_files_null = []

for batch_idx in range(n_batches_null):
    start = batch_idx * MAX_REQUESTS_PER_BATCH
    end = min(start + MAX_REQUESTS_PER_BATCH, n_posts_null)

    df_batch = df_null.iloc[start:end]
    jsonl_path = Path(OUTPUT_DIR_NULL) / f"reprocess_batch_{batch_idx:03d}.jsonl"

    with open(jsonl_path, "w", encoding="utf-8") as f:
        for idx, row in df_batch.iterrows():
            request = {
                "custom_id": str(idx),  # Use original index from the dataset
                "method": "POST",
                "url": "/v1/chat/completions",
                "body": {
                    "model": MODEL_NAME,
                    "messages": [
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user", "content": row["titlencontent"]}
                    ],
                    "max_completion_tokens": 1000
                }
            }
            f.write(json.dumps(request) + "\n")

    jsonl_files_null.append((batch_idx, jsonl_path, start, end - 1))
    print(f"Wrote {jsonl_path} ({len(df_batch)} requests)")

print(f"Created {len(jsonl_files_null)} JSONL files for reprocessing")

In [ ]:
# -------------------------------------
# UPLOAD REPROCESSING FILES
# -------------------------------------

uploaded_files_null = {}

for batch_idx, path, start, end in jsonl_files_null:
    with open(path, "rb") as f:
        uploaded = client.files.create(file=f, purpose="batch")

    uploaded_files_null[batch_idx] = {
        "file_id": uploaded.id,
        "row_start": start,
        "row_end": end,
        "input_file": str(path)
    }

    print(f"Uploaded reprocess batch {batch_idx}: file_id={uploaded.id}")

print(f"Uploaded {len(uploaded_files_null)} files for reprocessing")

In [ ]:
# -------------------------------------
# SUBMIT REPROCESSING BATCHES
# -------------------------------------

reprocess_batch_registry = []

for batch_idx, info in uploaded_files_null.items():
    batch = client.batches.create(
        input_file_id=info["file_id"],
        endpoint="/v1/chat/completions",
        completion_window="24h"
    )

    entry = {
        "batch_index": batch_idx,
        "batch_id": batch.id,
        "input_file": info["input_file"],
        "row_start": info["row_start"],
        "row_end": info["row_end"],
        "type": "reprocess"
    }

    reprocess_batch_registry.append(entry)
    print(f"Submitted reprocess batch {batch.id} for rows {info['row_start']}–{info['row_end']}")

# Save reprocess registry
reprocess_registry_path = Path(OUTPUT_DIR_NULL) / "reprocess_batch_registry.json"
with open(reprocess_registry_path, "w") as f:
    json.dump(reprocess_batch_registry, f, indent=2)

print(f"Reprocess batch registry saved to {reprocess_registry_path}")

In [ ]:
# -------------------------------------
# CHECK REPROCESSING BATCH STATUS
# -------------------------------------

print("📊 REPROCESSING BATCH STATUS")
print("=" * 60)

for entry in reprocess_batch_registry:
    batch = client.batches.retrieve(entry["batch_id"])
    
    print(f"📊 Reprocess Batch {entry['batch_index']} (ID: {batch.id})")
    print(f"   Status: {batch.status}")
    print(f"   Model: {batch.model}")
    print(f"   Created: {pd.to_datetime(batch.created_at, unit='s').strftime('%Y-%m-%d %H:%M:%S')}")
    
    if batch.expires_at:
        expires = pd.to_datetime(batch.expires_at, unit='s').strftime('%Y-%m-%d %H:%M:%S')
        print(f"   Expires: {expires}")
    
    if batch.request_counts:
        counts = batch.request_counts
        print(f"   Progress: {counts.completed}/{counts.total} completed, {counts.failed} failed")
    
    if batch.completed_at:
        completed = pd.to_datetime(batch.completed_at, unit='s').strftime('%Y-%m-%d %H:%M:%S')
        print(f"   Completed: {completed}")
    
    if batch.usage and batch.usage.total_tokens > 0:
        usage = batch.usage
        print(f"   Token Usage: {usage.input_tokens} input + {usage.output_tokens} output = {usage.total_tokens} total")

    print("   " + "-" * 50)

In [ ]:
# -------------------------------------
# DOWNLOAD REPROCESSING RESULTS
# -------------------------------------

reprocess_results = {}

for entry in reprocess_batch_registry:
    batch = client.batches.retrieve(entry["batch_id"])

    if batch.status != "completed":
        print(f"Reprocess batch {entry['batch_index']} not completed yet")
        continue

    file_id = batch.output_file_id
    content = client.files.content(file_id).text

    output_path = Path(OUTPUT_DIR_NULL) / f"reprocess_batch_output_{entry['batch_index']}.jsonl"
    output_path.write_text(content, encoding="utf-8")

    print(f"Downloaded reprocess results for batch {entry['batch_index']}")

print("All completed reprocess batches downloaded")

In [ ]:
# -------------------------------------
# MERGE REPROCESSING RESULTS
# -------------------------------------

# Parse reprocessing results
reprocess_results = {}
failed_responses = []

for path in Path(OUTPUT_DIR_NULL).glob("reprocess_batch_output_*.jsonl"):
    with open(path, "r") as f:
        for line in f:
            record = json.loads(line)
            if record.get("error") is None:
                idx = record["custom_id"]
                response_body = record["response"]["body"]
                content = response_body["choices"][0]["message"]["content"]
                finish_reason = response_body["choices"][0]["finish_reason"]
                
                # Check if the response is complete and has content
                if finish_reason == "stop" and content.strip():
                    reprocess_results[int(idx)] = content
                elif finish_reason == "length":
                    failed_responses.append((idx, "Token limit exceeded"))
                elif not content.strip():
                    failed_responses.append((idx, "Empty content"))
                else:
                    failed_responses.append((idx, f"Finish reason: {finish_reason}"))

print(f"Recovered {len(reprocess_results)} reprocessed classifications")
print(f"Failed responses: {len(failed_responses)}")
if failed_responses:
    print("First few failed responses:")
    for idx, reason in failed_responses[:5]:
        print(f"  Index {idx}: {reason}")

# Parse the reprocessed JSON results
def safe_json_parse(x):
    """Safely parse JSON with error handling"""
    if pd.isnull(x) or not x.strip():
        return {}
    try:
        return json.loads(x)
    except json.JSONDecodeError as e:
        print(f"JSON parsing error for content: {str(x)[:100]}...")
        return {}

# Apply reprocessed results to null rows
reprocessed_data = {}
for idx, classification in reprocess_results.items():
    parsed = safe_json_parse(classification)
    if parsed:  # Only update if parsing was successful
        reprocessed_data[idx] = parsed

print(f"Successfully parsed {len(reprocessed_data)} reprocessed classifications")

In [ ]:
# -------------------------------------
# UPDATE ORIGINAL DATASET WITH REPROCESSED RESULTS
# -------------------------------------

# Load the original dataset with all results
df_final = pd.read_csv(OUTPUT_CSV)
print(f"Original dataset shape: {df_final.shape}")

# Count null values before update
null_before = df_final['sentiment_openai'].isnull().sum()
print(f"Null sentiment_openai values before update: {null_before}")

# Update null values with reprocessed results
updated_count = 0
for idx in reprocessed_data:
    if idx in df_final.index and pd.isnull(df_final.loc[idx, 'sentiment_openai']):
        # Update all classification columns
        for key, value in reprocessed_data[idx].items():
            col_name = f"{key}_openai"
            if col_name in df_final.columns:
                df_final.loc[idx, col_name] = value
        updated_count += 1

print(f"Updated {updated_count} rows with reprocessed classifications")

# Count null values after update
null_after = df_final['sentiment_openai'].isnull().sum()
print(f"Null sentiment_openai values after update: {null_after}")

# Save the updated dataset
df_final.to_csv(OUTPUT_CSV, index=False)
print(f"Updated dataset saved to {OUTPUT_CSV}")

# Summary
print("\n📊 REPROCESSING SUMMARY")
print("=" * 40)
print(f"Rows reprocessed: {updated_count}")
print(f"Remaining null values: {null_after}")
print(f"Success rate: {((null_before - null_after) / null_before * 100):.1f}%" if null_before > 0 else "N/A")